# Complete End-to-End Workflow

This notebook documents the full machine learning workflow for the crop-yield challenge: raw data ingestion, cleaning, feature engineering, model training, evaluation, and final export.

In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd

from cleaning import clean_plot_data, clean_weather_data, clean_market_price, build_cleaning_log
from features import add_weather_features, attach_price_analysis, build_feature_columns, make_data_dictionary
from train import build_model_data, build_preprocessor, fit_and_eval

print(f'ROOT: {ROOT}')
print('Ready to run the full crop-yield workflow.')

In [ ]:
# Load raw data
raw_train = pd.read_csv(ROOT / 'data' / 'raw' / 'crop_yield_train.csv')
raw_test = pd.read_csv(ROOT / 'data' / 'raw' / 'crop_yield_leaderboard_test.csv')
raw_weather = pd.read_csv(ROOT / 'data' / 'raw' / 'regional_weather.csv')
raw_prices = pd.read_csv(ROOT / 'data' / 'raw' / 'market_prices.csv')

print('Train rows:', len(raw_train))
print('Test rows:', len(raw_test))
print('Weather rows:', len(raw_weather))
print('Price rows:', len(raw_prices))
print(raw_train.head(3))

In [ ]:
# Step 1: Clean raw tables
train_clean, test_clean, train_log = clean_plot_data(raw_train, raw_test)
weather_clean, weather_log = clean_weather_data(raw_weather)
price_clean, price_log = clean_market_price(raw_prices)
cleaning_log = build_cleaning_log(train_log, weather_log, price_log)

print('Train clean shape:', train_clean.shape)
print('Test clean shape:', test_clean.shape)
print('Weather clean shape:', weather_clean.shape)
print('Market price clean shape:', price_clean.shape)
print(cleaning_log.head())

In [ ]:
# Step 2: Build feature-engineered data
feature_train = add_weather_features(train_clean, weather_clean)
feature_test = add_weather_features(test_clean, weather_clean)

feature_train = attach_price_analysis(feature_train, price_clean)
feature_test = attach_price_analysis(feature_test, price_clean)

feature_train['year'] = feature_train['survey_year']
feature_test['year'] = feature_test['survey_year']

feature_cols = build_feature_columns()
for col in feature_cols:
    if col not in feature_train.columns:
        feature_train[col] = 0.0
    if col not in feature_test.columns:
        feature_test[col] = 0.0

print('Feature set columns:', len(feature_cols))
print(feature_train[['survey_year', 'planting_month', 'season_mean_temp', 'season_rainfall_total', 'yield_tons_per_ha']].head())

In [ ]:
# Step 3: Save processed master files
train_master = feature_train.copy()
test_master = feature_test.copy()

train_master.to_csv(ROOT / 'data' / 'processed' / 'master_train.csv', index=False)
test_master.to_csv(ROOT / 'data' / 'processed' / 'master_test.csv', index=False)

data_dictionary = make_data_dictionary(train_master)
data_dictionary.to_csv(ROOT / 'data' / 'processed' / 'data_dictionary_master.csv', index=False)

print('Saved processed files:')
for path in [
    ROOT / 'data' / 'processed' / 'master_train.csv',
    ROOT / 'data' / 'processed' / 'master_test.csv',
    ROOT / 'data' / 'processed' / 'data_dictionary_master.csv',
]:
    print(' -', path)

In [ ]:
# Step 4: Prepare train/validation split and model pipeline
train_df = pd.read_csv(ROOT / 'data' / 'processed' / 'master_train.csv')
feature_cols = build_feature_columns()
X = train_df[feature_cols].copy()
y = train_df['yield_tons_per_ha'].copy()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
preprocessor = build_preprocessor(X_train)

print('X_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)
print('Target mean:', y_train.mean())

In [ ]:
# Step 5: Train and compare candidate models
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

models = {
    'linear_regression': LinearRegression(),
    'random_forest': RandomForestRegressor(random_state=42, n_estimators=150, max_depth=12, n_jobs=-1),
}

results = []
for name, model in models.items():
    pipe = __import__('sklearn').pipeline.Pipeline([
        ('prep', preprocessor),
        ('model', model),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    results.append({
        'model': name,
        'rmse': np.sqrt(mean_squared_error(y_val, pred)),
        'mae': mean_absolute_error(y_val, pred),
        'r2': r2_score(y_val, pred),
    })

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
# Step 6: Final model fit and export
final_model = RandomForestRegressor(random_state=42, n_estimators=150, max_depth=12, n_jobs=-1)
final_pipe = __import__('sklearn').pipeline.Pipeline([
    ('prep', preprocessor),
    ('model', final_model),
])
final_pipe.fit(X_train, y_train)

# Save model artifact
joblib = __import__('joblib')
joblib.dump(final_pipe, ROOT / 'models' / 'final_model.joblib')

# Predict on leaderboard test set
master_test = pd.read_csv(ROOT / 'data' / 'processed' / 'master_test.csv')
test_X = master_test[feature_cols].copy()
submission = pd.DataFrame({
    'plot_id': master_test['plot_id'],
    'predicted_yield_tons_per_ha': final_pipe.predict(test_X)
})
submission.to_csv(ROOT / 'submission' / 'team_beso_submission.csv', index=False)

print('Saved final model and submission CSV.')
print(submission.head())

## Workflow summary

This workflow covered the full path from raw source files to the final model and prediction export. The crucial compliance rule is that the raw data stays untouched, while all transformations are reproduced in code and written into the processed data folder. Weather-derived seasonal features were used as genuine model inputs, and price information was kept out of the predictive feature set.